# A4 — Huấn luyện BRFSS 2015 theo Protocol V2

Notebook là giao diện điều khiển script A4, không sao chép thuật toán vào nhiều nơi. Chỉ chạy với CSV Kaggle gốc; dữ liệu `--quick` là smoke.

## 1. Kiểm tra thư mục và file CSV

Mở từ repository root, thư mục `research/lead_modeling/a4_a5/` hoặc `notebooks/`. Cell đầu tự tìm component nested; nếu dùng Colab/Kaggle, mount cả component và chỉnh `ROOT`/`DATA` theo vị trí thực tế.

In [ ]:
from pathlib import Path
# Resolve the nested component from repository, component or notebooks directory.
_here = Path.cwd().resolve()
ROOT = None
for _base in (_here, *_here.parents):
    for _candidate in (_base, _base / 'research' / 'lead_modeling' / 'a4_a5'):
        if (_candidate / 'scripts' / 'run_a4.py').is_file() and (_candidate / 'config' / 'A2_protocol_v2.yaml').is_file():
            ROOT = _candidate
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository, A4/A5 component or notebooks directory')
DATA = ROOT / 'data' / 'diabetes_binary_health_indicators_BRFSS2015.csv'
assert DATA.is_file(), f'Chưa có file {DATA}'
print('Dataset:', DATA)

## 2. Kiểm tra dataset

Xác minh 21 đặc trưng, tên target và tỷ lệ dương trước khi train.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src'))
from diabetes_research.core import load_dataset, dataset_audit
X,y = load_dataset(DATA)
dataset_audit(X,y)

## 3. Chạy experiment (nhiều CV fit, có thể tốn thời gian)

Đổi tên OUTPUT nếu muốn chạy lại. Mặc định 10 trial/họ mô hình, 3 folds.

In [ ]:
import subprocess, sys
OUTPUT = ROOT / 'results' / 'a4_full_001'
cmd = [sys.executable, str(ROOT/'scripts'/'run_a4.py'),
       '--data',str(DATA),'--output-dir',str(OUTPUT), '--trials','10','--cv','3']
subprocess.run(cmd, check=True)

## 4. Xem kết quả thực nghiệm

Không chỉnh hyperparameter hoặc ngưỡng bằng kết quả test.

In [ ]:
import pandas as pd
display(pd.read_csv(OUTPUT/'cv_results.csv').head())
display(pd.read_csv(OUTPUT/'validation_metrics.csv'))
display(pd.read_csv(OUTPUT/'test_metrics.csv'))

In [ ]:
import json
print(json.dumps(json.loads((OUTPUT/'selection.json').read_text()),indent=2,ensure_ascii=False))